# Calvin Site 1 optimization replay

Inspect the frozen choice, matched-cylinder losses and fitted schema-4 input.
The candidate was selected on fitting cores. The comparison cells use fresh simulation seeds at previously inspected cores.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import os
import subprocess
from textwrap import dedent
import pandas as pd
from IPython.display import Image, display

project = next(p for p in (Path.cwd(), *Path.cwd().parents) if
               (p / '.runtime/calvin-site1-fit-20261119/PROTOCOL.json').is_file())
study = project / '.runtime/calvin-site1-fit-20261119'
protocol = json.loads((study / 'PROTOCOL.json').read_text())
for relative_path, expected in protocol['files'].items():
    assert sha256((project / relative_path).read_bytes()).hexdigest() == expected
selection = json.loads((study / 'selection.json').read_text())
assert sha256((study / 'development_ranks.json').read_bytes()).hexdigest() == selection['development_ranks_sha256']
display(pd.read_csv(study / 'development_ranks.csv')[['name', 'spatial_score', 'rld_depth_tv', 'mass_depth_tv']])

In [ ]:
summary = json.loads((study / 'comparison_summary.json').read_text())
settings = pd.read_csv(study / 'parameter_values.csv')
metrics = pd.read_csv(study / 'spatial_scores.csv').query('split == "Site1_hold"')
view = ['cutoff_mm', 'qc', 'case', 'seeds', 'spatial_score', 'rld_nmae', 'mass_nmae', 'rld_depth_tv', 'mass_depth_tv']
display(metrics[view].round(3))
display(settings)
display(Image(filename=str(study / 'held_core_profiles.png'), width=970))

## Reproduce the score and simulate the example

The Python 3.10 environment and the source snapshot are recorded in the protocol.
Set `REPLAY_SCORES = True` to verify the saved comparison receipts and recompute the scores.
Set `RUN_EXAMPLE = True` to simulate the example seed and check both cutoff projections against the saved run.
This uses one additional local simulation call. It can take a minute or more.

In [ ]:
REPLAY_SCORES = False
RUN_EXAMPLE = False
launch_dir = project / '.runtime/spatial-construction-20261119/source'
python310 = project / '.runtime/envs/py310-qualification/bin/python'
env = dict(os.environ, PYTHONPATH=str(launch_dir), PYTHONDONTWRITEBYTECODE='1',
           OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
if REPLAY_SCORES:
    subprocess.run([python310, study / 'results.py'], check=True, cwd=launch_dir, env=env)
if RUN_EXAMPLE:
    program = dedent("""
        import sys
        import numpy as np
        sys.path.insert(0, sys.argv[1])
        import search
        import results
        search.freeze()
        _, panels, _, _ = results.panels()
        _, _, tubes, offsets = search.observation_support()
        config = search.cutoff.read(search.HERE / results.CANDIDATE)
        expected = next(record for record in panels['challenger'] if record['seed'] == config['random_seed'])
        sim = search.RootSystemSimulation(random_seed=config['random_seed'])
        sim.run(config)
        for threshold in search.cutoff.CUTOFFS:
            predicted = search.cutoff.observe(sim, threshold, tubes, offsets).set_index('target_row')[search.SCORER.PREDICTIONS]
            np.testing.assert_array_equal(predicted.to_numpy(), expected['predictions'][str(threshold)])
        print(sim.result.metadata['root_summary'])
    """)
    subprocess.run([python310, '-c', program, study], check=True, cwd=launch_dir, env=env)
print('Input file', study / 'Calvin_Site1_candidate.schema4.private.json')
print('To replay all search stages from saved receipts, run search.py from the source snapshot with the same Python and environment.')